In [ ]:
import monai
from monai.networks.nets import DynUNet
import cv2
import os
import numpy as np
from monai.transforms import (
    Compose, LoadImaged,
    EnsureChannelFirstd, ScaleIntensityd,
)
from monai.data import DataLoader, Dataset, list_data_collate
from monai.metrics import DiceMetric
from monai.visualize import plot_2d_or_3d_image
from pathlib import Path
import torch
from torch.utils.tensorboard import SummaryWriter

In [ ]:
train_dir = '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/train'
validation_dir = '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/validation'
test_dir = '/work/cvcs2026/LZMM/OpenEDS/openEDS/openEDS/test'

In [ ]:
train_images_dir = Path(os.path.join(train_dir, "images"))
train_masks_dir = Path(os.path.join(train_dir, "masks"))

images = sorted(train_images_dir.glob("*.png"))

data = [
    {
        "image": str(img),
        "mask": str(train_masks_dir / img.name)
    }
    for img in images
]

print(data)

In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["image", "mask"]),
    EnsureChannelFirstd(keys=["image", "mask"]),
])

train_ds = Dataset(
    data=data,
    transform=train_transforms
)

train_loader = DataLoader(
    train_ds,
    batch_size=1024,
    shuffle=True,
    num_workers=8,
    collate_fn=list_data_collate,
    pin_memory=torch.cuda.is_available(),
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DynUNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    strides = (1, 2, 2, 2, 2, 2)
).to(device)

loss_function = monai.losses.DiceLoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-3)

In [ ]:
# start a typical PyTorch training
val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = list()
metric_values = list()
writer = SummaryWriter()
for epoch in range(10):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{10}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data["img"].to(device), batch_data["seg"].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

print(f"train completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()